# wf_v4.3 real runs with Llama (diary Nov 13): the 8B design
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v4.3` (from `wf-v4.2-llama`), folder `step1a/`. Results in `results/wf_v43/`.

Model: Llama-3.1-8B-Instruct, bf16, vLLM, temperature 0, guided JSON from each job's schema (setting `llama8b`). Runtime: **A100 (40 GB or more)**.
Idea: Llama gets small, closed questions; code does the bookkeeping. Prompts of U0 and U2 were built on the laptop and are in the branch.

- Cells 1-5: setup (as the Nov 16 / wf_v4.2 notebooks).
- Cell 6: data check (bge-m3 vectors and GT files the post steps need).
- Cell 7: **U0** baseline: frozen split pass -> reconcile -> duplicate check on the wf_v4.2 option B A answers, then the L2 score. About 160 calls, about 5 min (estimate).
- Cell 8: **U2** Step 3 v4.3: one name per cluster, batches of 5, 3 orders + vote, worked examples, then the same post steps; writes the Step 4 input `a_llama43`. About 300 calls, about 10-15 min (estimate).
- Cell 9: **U4** Step 4 v4.3 on inputs a, c1 and a_llama43 (dev50 + dev50b, seeds 0-2, 18 chains; at most 38 short calls per chain), then assemble and score. About 20-30 min (estimate).
- Each cell commits and pushes its results folder. If time runs short, the order of priority is cell 7, 8, then 9.
- After a crash or disconnect: rerun cells 1, 3, 4, 5 and then the cell you were on. Answered calls are read from the cache.
- Colab Secrets `HF_TOKEN` and `GH_TOKEN`, notebook access on. Never put a token in a cell.
- When the GPU work is done: **Runtime > Disconnect and delete runtime**, so the paid GPU stops.


In [ ]:
#@title 1. Tokens, clone/pull wf-v4.3, git identity
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
BRANCH = "wf-v4.3"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** the install prints dependency-resolver warnings; ignore them. Then **Runtime > Restart session**, rerun cell 1 and skip cell 2.
gliner and sentence-transformers are installed with `--no-deps` so they do not replace the torch / transformers that vLLM needs. Cell 4 checks that they load.

In [ ]:
#@title 2. Install (pinned; a few minutes; then restart the session and skip this cell)
!pip install -q vllm==0.33.0
!pip install -q --no-deps gliner==0.2.29 sentence-transformers==5.6.0
!pip install -q sentencepiece

In [ ]:
#@title 3. Model setting (config/wf_v41_models.json) and GPU check
SETTING = "llama8b"  #@param ["llama8b"]
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from s1a import nov16_v41 as NV
SET = NV.model_setting(SETTING)
print(SETTING, SET)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GiB")
assert gb >= SET["min_gpu_gb"], f"only {gb:.1f} GiB: {SETTING} needs {SET['min_gpu_gb']} GiB (an A100)"

In [ ]:
#@title 4. Load the model (gpu_memory_utilization leaves room for GLiNER and bge-m3 of loop 2, T6)
MODEL, REVISION = SET["model"], SET["revision"]   # from the setting of cell 3
GPU_UTIL = 0.80  #@param {type:"number"}
MAX_LEN = 32768
import vllm
from vllm import LLM, SamplingParams
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization=SET["quantization"],
          max_model_len=MAX_LEN, gpu_memory_utilization=GPU_UTIL)

In [ ]:
#@title 5. Backend: temperature 0, guided JSON per job schema, batches, cache per job (s1a/nov16_v41.VllmBackend)
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)
backend = NV.VllmBackend(llm, MODEL, REVISION, sampling, max_len=MAX_LEN, batch=BATCH, vllm_version=vllm.__version__)
backend.model_id

In [ ]:
#@title 6. Data check: bge-m3 vectors (reconcile) and GT levels (score)
from pathlib import Path
need = ["results/graph_v22_dev50/spans/vectors.npz", "results/graph_v22_dev50b/spans/vectors.npz",
        "results/loop1_v35/gt_levels.csv", "results/loop1_v37/dev50b/gt/gt_levels.csv",
        "results/wf_v43/u0/manifest.json", "results/wf_v43/u2/manifest.json"]
missing = [p for p in need if not Path(p).exists()]
print("missing:", missing or "none")
assert not missing, "files missing: stop and tell Claude"


In [ ]:
#@title 7. U0: frozen Step 3 post steps on Llama option B answers (split -> reconcile -> dupcheck), then the L2 score
import sys, subprocess
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import u0_step3_post_v43 as U0
print(U0.run(U0.OUT, backend))
!python scripts/u0_step3_post_v43.py --score
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v43 U0 step3 post", "wf_v43/u0"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("U0 done")


In [ ]:
#@title 8. U2: Step 3 v4.3 (names in batches of 5, 3 orders + vote, examples, post steps); writes a_llama43
import sys, subprocess
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import u2_step3_v43 as U2
print(U2.run(U2.OUT, backend))
res, sizes = U2.score_all(U2.OUT)
print("a_llama43:", sizes)
print((U2.OUT / "scores" / "summary.md").read_text(encoding="utf-8")[:3000])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v43 U2 step3 v43", "wf_v43/u2"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("U2 done")


In [ ]:
#@title 9. U4: Step 4 v4.3 on a, c1, a_llama43 (18 chains), assemble all, score
import subprocess
from pathlib import Path
from s1a import step4_v43 as S
for inp in S.INPUTS:                      # a, c1, a_llama43 (priority order); a_llama43 needs cell 8
    print(inp, S.run(S.OUT, backend, inputs=(inp,)))
    p = subprocess.run(["bash", "colab/sync.sh", "push-results", f"wf_v43 U4 step4 v43 {inp}", "wf_v43/u4"], capture_output=True, text=True)
    print(p.stdout.strip() or p.stderr[-500:])
S.assemble_all(S.OUT, S.make_chains(S.INPUTS), S.ROOT)     # final/ and repairs.json over all 18 chains
r = subprocess.run(["python", "scripts/u4_step4_v43.py", "--score", "--out", "results/wf_v43/u4"], capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-1500:])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v43 U4 scores", "wf_v43/u4"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("U4 done: tell Claude 'wf_v4.3 run done'")
